# Tarefa 4 - Random Forest com valores ausentes e K-means

Quarta tarefa avaliada da disciplina **Técnicas e Algoritmos em Ciência de Dados**.

Esta versão tem duas partes:

1. [Random Forest com valores ausentes](#Classification) (60%)
2. [Agrupamento com K-means](#K-Means) (40%)

Use o mesmo identificador aleatório de 6 dígitos usado nas tarefas anteriores e escreva-o na primeira célula do seu notebook submetido.

Você pode usar `numpy`, `pandas`, `matplotlib` e métricas de avaliação como AUROC. Você **não pode** usar implementações prontas de árvore de decisão, Random Forest, K-means ou ferramentas prontas de busca automática de parâmetros. Qualquer função auxiliar deve estar dentro deste notebook.


In [ ]:
import math

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    PLOTTING_AVAILABLE = False
    print("matplotlib não está instalado neste ambiente; ele será necessário para gerar os gráficos pedidos.")

try:
    display
except NameError:
    def display(obj):
        print(obj)


def roc_auc_score(y_true, y_score):
    """Calcula AUROC para rótulos binários, com tratamento correto de empates."""
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    positive = y_true == 1
    negative = y_true == 0
    n_pos = int(positive.sum())
    n_neg = int(negative.sum())
    if n_pos == 0 or n_neg == 0:
        raise ValueError("AUROC requer exemplos das duas classes.")

    order = np.argsort(y_score, kind="mergesort")
    sorted_scores = y_score[order]
    ranks = np.empty(len(y_score), dtype=float)
    start = 0
    while start < len(y_score):
        end = start + 1
        while end < len(y_score) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        average_rank = (start + 1 + end) / 2.0
        ranks[order[start:end]] = average_rank
        start = end

    rank_sum_pos = ranks[positive].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


<a id="Classification"></a>
## Parte 1 - Random Forest com valores ausentes (valor: 60%)

Você usará o arquivo `horse_colic_missing.csv`, baseado no conjunto Horse Colic. O alvo é `surgical_lesion`, em que `1` indica que o caso tinha lesão cirúrgica e `0` indica que não tinha.

Use apenas os atributos clínicos pré-desfecho listados abaixo. Não use `surgery`, `hospital_number`, `outcome`, códigos de lesão, `cp_data` ou outros campos que funcionem como identificadores, tratamento, desfecho ou vazamento de informação.

### Objetivo

Você pode reutilizar e adaptar suas implementações anteriores de árvore de decisão e Random Forest. O foco desta tarefa é modificar as árvores da floresta para lidar com valores ausentes durante a escolha das divisões, durante o treinamento e durante a predição.

### Estratégias obrigatórias para valores ausentes

Compare Florestas Aleatórias que diferem principalmente na forma como suas árvores tratam valores ausentes:

1. **Baseline simples de pré-processamento**: implemente uma referência simples, como imputação por mediana/moda aprendida apenas no treino. Essa estratégia serve como baseline.
2. **Roteamento fracionário por frequência de ramo**: ao avaliar uma divisão em um nó, use apenas os exemplos com valor observado naquele atributo para estimar as probabilidades dos ramos:

   ```text
   p_left = n_left_observed / n_observed
   p_right = n_right_observed / n_observed
   ```

   Os exemplos com valor ausente devem ser enviados para os dois filhos com esses pesos. A entropia, o ganho de informação e a frequência das classes nas folhas devem usar pesos: sempre que uma fórmula envolver contagem de exemplos, substitua a contagem pela soma dos pesos.

### O que entregar nesta parte

1. Carregue o conjunto de dados e descreva a fração de valores ausentes em cada atributo.
2. Faça a divisão treino/validação/teste com proporção 70/15/15 e `seed = 13`.
3. Adapte a árvore de decisão para usar pesos de amostras em entropia, ganho de informação e probabilidade da classe positiva nas folhas.
4. Armazene em cada nó interno as probabilidades de ramo estimadas a partir dos exemplos observados.
5. Implemente a predição para exemplos com valores ausentes usando o roteamento fracionário por frequência de ramo.
6. Implemente a Random Forest com bootstrap, subconjunto aleatório de atributos e agregação por média das probabilidades positivas ou média dos votos binários.
7. Escolha manualmente poucas combinações de hiperparâmetros usando AUROC na validação. Use `num_trees` entre 5 e 20 e profundidade máxima não superior a 5.
8. Treine novamente com treino + validação usando a melhor combinação de parâmetros e reporte acurácia com limiar 0.5 e AUROC no teste.
9. Compare as duas estratégias em uma tabela final e discuta qual funcionou melhor.


In [ ]:
SEED = 13
TARGET = "surgical_lesion"
FEATURE_COLUMNS = [
    "age",
    "rectal_temperature",
    "pulse",
    "respiratory_rate",
    "temperature_of_extremities",
    "peripheral_pulse",
    "mucous_membranes",
    "capillary_refill_time",
    "pain",
    "peristalsis",
    "abdominal_distension",
    "nasogastric_tube",
    "nasogastric_reflux",
    "nasogastric_reflux_ph",
    "rectal_exam_feces",
    "abdomen",
    "packed_cell_volume",
    "total_protein",
    "abdominocentesis_appearance",
    "abdomcentesis_total_protein",
]
NUMERIC_COLUMNS = {
    "rectal_temperature",
    "pulse",
    "respiratory_rate",
    "nasogastric_reflux_ph",
    "packed_cell_volume",
    "total_protein",
    "abdomcentesis_total_protein",
}
CATEGORICAL_COLUMNS = [col for col in FEATURE_COLUMNS if col not in NUMERIC_COLUMNS]


df = pd.read_csv("horse_colic_missing.csv")
display(df.head())
missing_summary = df[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
display(missing_summary)


In [ ]:
def stratified_split(df, target=TARGET, train_size=0.70, val_size=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for _, group in df.groupby(target):
        idx = group.index.to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(round(train_size * n))
        n_val = int(round(val_size * n))
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:n_train + n_val])
        test_idx.extend(idx[n_train + n_val:])
    return (
        df.loc[train_idx].sample(frac=1, random_state=seed).reset_index(drop=True),
        df.loc[val_idx].sample(frac=1, random_state=seed + 1).reset_index(drop=True),
        df.loc[test_idx].sample(frac=1, random_state=seed + 2).reset_index(drop=True),
    )


In [ ]:
train_df, val_df, test_df = stratified_split(df)
train_y = train_df[TARGET].to_numpy(dtype=int)
val_y = val_df[TARGET].to_numpy(dtype=int)
test_y = test_df[TARGET].to_numpy(dtype=int)

print("Tamanhos treino/validação/teste:", len(train_df), len(val_df), len(test_df))
print("Distribuição do alvo no treino:", train_df[TARGET].value_counts(normalize=True).sort_index().to_dict())


### Árvore de decisão ponderada

Implemente abaixo a adaptação da árvore de decisão. Sua implementação deve incluir:

- entropia ponderada;
- ganho de informação ponderado;
- probabilidade positiva ponderada em cada folha;
- escolha de divisões ignorando valores ausentes apenas para calcular as frequências observadas dos ramos;
- envio fracionário dos exemplos ausentes para os dois filhos;
- armazenamento de `p_left` e `p_right` em cada nó interno para uso na predição.


In [ ]:
# Escreva aqui a implementação da árvore de decisão ponderada e do
# roteamento fracionário por frequência de ramo.


### Random Forest e comparação das estratégias

Implemente abaixo a Random Forest. A comparação final deve isolar principalmente a estratégia de tratamento de valores ausentes:

- baseline com imputação mediana/moda aprendida apenas no treino;
- árvore com roteamento fracionário por frequência de ramo.

Use bootstrap para cada árvore e um subconjunto aleatório de atributos por árvore. Escolha manualmente alguns valores para `num_trees`, `max_depth` e outros critérios de parada usando AUROC na validação. Depois, treine com treino + validação e avalie no teste.


In [ ]:
# Escreva aqui a Random Forest, a busca manual por hiperparâmetros,
# a comparação na validação e a avaliação final no conjunto de teste.


<a id="K-Means"></a>
## Parte 2 - Agrupamento com K-means (valor: 40%)

Nesta parte, implemente K-means do zero usando os pontos artificiais abaixo. O objetivo é visualizar o movimento dos centroides e analisar a soma das distâncias quadradas.

Requisitos:

1. Use `K = 3`, `epsilon = 1e-4`, `max_iter = 150` e seed `13`.
2. Inicialize os centroides escolhendo `K` pontos do conjunto de dados sem reposição.
3. Em cada iteração:
   - atribua cada ponto ao centroide mais próximo usando distância euclidiana;
   - atualize cada centroide para a média dos pontos atribuídos ao cluster;
   - se um cluster não receber pontos, mantenha o centroide anterior desse cluster;
   - calcule e armazene a soma das distâncias quadradas.
4. Use a norma euclidiana do deslocamento conjunto dos centroides como critério de convergência. Pare quando ela for menor que `epsilon`.
5. Armazene o histórico de centroides, rótulos de cluster e soma das distâncias quadradas.
6. Plote 5 momentos determinísticos: estado inicial antes do aprendizado, estado final e três estados intermediários igualmente espaçados. Se houver menos de 5 estados, repita estados disponíveis sem inventar novas iterações.
7. Plote também a soma das distâncias quadradas ao longo das iterações.


In [ ]:
K = 3
EPSILON = 1e-4
MAX_ITER = 150

np.random.seed(SEED)
num_samples = 200
num_features = 2
X = np.random.randn(num_samples, num_features) * 1.5 + np.array([[2, 2]])
X = np.concatenate([X, np.random.randn(num_samples, num_features) * 3 + np.array([[-5, -5]])])
X = np.concatenate([X, np.random.randn(num_samples, num_features) * 2 + np.array([[7, -5]])])

print(X.shape)


In [ ]:
# Escreva aqui sua implementação de K-means, os gráficos dos 5 momentos
# e o gráfico da soma das distâncias quadradas.
